# Real-Time Machine Learning with Apache Spark Streaming

# Stream Processing Production Pipeline
- **Module**: Distributed Machine Learning
- **Dataset**: [bank.csv](https://drive.google.com/file/d/17dekep3p_WYdegUvSmVK8vRaGwOguJiT/view?usp=sharing)

# Github Link
https://github.com/saiabhiram-simhadri/financial-forecasting-distributed-ml

# Spark Streaming

## Initial Setup

In [1]:
import textwrap

def custom_display(
    title: str | None = None,
    content: str | list[str] | None = None,
    max_width: int = 70,
    padding: int = 2
):
    """
    Renders structured text inside a Unicode box.
    - Centers the title/header if the content width spans wider.
    - Supports (title, content), (content), or (title) dynamically.
    - Honors explicit '\n' line breaks and wraps unbroken lines exceeding max_width.
    """
    pad_space = " " * padding

    # Handle single positional argument intended as content
    if content is None and title is not None and ('\n' in title or len(title) > max_width):
        content = title
        title = None

    # Parse and wrap content lines
    formatted_lines = []
    if content is not None:
        if isinstance(content, str):
            raw_sections = content.split('\n')
        else:
            raw_sections = [line for item in content for line in str(item).split('\n')]

        for section in raw_sections:
            stripped = section.strip()
            if not stripped:
                formatted_lines.append("")
            elif len(section) > max_width:
                formatted_lines.extend(textwrap.wrap(section, width=max_width))
            else:
                formatted_lines.append(section)

    title_str = title.strip() if title else ""
    has_title = bool(title_str)
    has_content = bool(formatted_lines)

    if not has_title and not has_content:
        return

    # Calculate max width across both title and content
    max_content_len = max([len(line) for line in formatted_lines]) if has_content else 0
    max_line_len = max(len(title_str) if has_title else 0, max_content_len)

    inner_width = max_line_len + (padding * 2)

    # Render Box
    print(f"┌{'─' * inner_width}┐")
    if has_title:
        # Centers the header text across the total inner width
        print(f"│{title_str.center(inner_width)}│")
        if has_content:
            print(f"├{'─' * inner_width}┤")

    if has_content:
        for line in formatted_lines:
            print(f"│{pad_space}{line.ljust(max_line_len)}{pad_space}│")

    print(f"└{'─' * inner_width}┘")

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Suppress non-critical warnings
warnings.filterwarnings("ignore")

%config InlineBackend.figure_format = 'retina'

CUSTOM_THEME = {
    'bg':       '#18191C',  # Soft Dark Slate Gray
    'card':     '#222428',  # Elevated Surface Card
    'grid':     '#32363C',  # Low-Contrast Grid Lines
    'text':     '#E3E5E8',  # Warm Off-White
    'subtext':  '#9B9D9E',  # Muted Axis Label Grey
    'teal':     '#2DD4BF',  # Soft Mint Teal
    'coral':    '#F87171',  # Soft Crimson Coral
    'purple':   '#A78BFA',  # Soft Amber Gold
    'orange':   '#FB923C',  # Soft Warm Orange
}

# Apply global parameters
plt.rcParams.update({
    'figure.facecolor':  CUSTOM_THEME['bg'],
    'axes.facecolor':    CUSTOM_THEME['card'],
    'savefig.facecolor': CUSTOM_THEME['bg'],
    'axes.edgecolor':    CUSTOM_THEME['grid'],
    'axes.labelcolor':   CUSTOM_THEME['text'],
    'xtick.color':       CUSTOM_THEME['subtext'],
    'ytick.color':       CUSTOM_THEME['subtext'],
    'grid.color':        CUSTOM_THEME['grid'],
    'grid.linestyle':    '--',
    'grid.alpha':        0.6,
    'text.color':        CUSTOM_THEME['text'],
})

## Environment Setup & Stable Spark Streaming Initialization
We configure OpenJDK 17 with the required JVM module flags and initialize a streaming-optimized local `SparkSession`.

In [3]:
# Clean Installation of Java 17 and PySpark
!apt-get update -qq > /dev/null
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install -q pyspark py4j findspark gdown pandas matplotlib seaborn

import os
import shutil
import time
from datetime import datetime

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# Set JVM module opens to avoid Py4J gateway errors on Java 17+
jvm_opens = [
    "--add-opens=java.base/java.lang=ALL-UNNAMED",
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED",
    "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED",
    "--add-opens=java.base/java.io=ALL-UNNAMED",
    "--add-opens=java.base/java.net=ALL-UNNAMED",
    "--add-opens=java.base/java.nio=ALL-UNNAMED",
    "--add-opens=java.base/java.util=ALL-UNNAMED",
    "--add-opens=java.base/java.util.concurrent=ALL-UNNAMED",
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED",
]
os.environ["PYSPARK_SUBMIT_ARGS"] = f'--driver-java-options "{" ".join(jvm_opens)}" pyspark-shell'

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("RealTime_Banking_Streaming_ML") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
custom_display("Apache Spark Streaming Session initialized successfully!"+
               f"\nSpark Version: {spark.version}")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
┌────────────────────────────────────────────────────────────┐
│  Apache Spark Streaming Session initialized successfully!  │
│  Spark Version: 4.0.4                                      │
└────────────────────────────────────────────────────────────┘


In [5]:
# Dataset Verification & Local Directory Setup
# Check or download bank.csv
if not os.path.exists("bank.csv"):
    import gdown
    CSV_FILE_ID = "17dekep3p_WYdegUvSmVK8vRaGwOguJiT"
    gdown.download(f"https://drive.google.com/uc?id={CSV_FILE_ID}", "bank.csv", quiet=False)

# Setup directories for file-based streaming simulation
STREAM_DATA_DIR = "/tmp/streaming_data_source"
STREAM_CHECKPOINT_DIR = "/tmp/streaming_checkpoints"

for d in [STREAM_DATA_DIR, STREAM_CHECKPOINT_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

print('\n')
custom_display(f"Streaming source directory ready: {STREAM_DATA_DIR}")



┌────────────────────────────────────────────────────────────────┐
│  Streaming source directory ready: /tmp/streaming_data_source  │
└────────────────────────────────────────────────────────────────┘


## Task 1: Stream Processing and Data Aggregation
- **Stream Generation:** Break `bank.csv` into micro-batches and feed them into a live file-based Spark stream including `age`, `job`, `balance`, `duration`, and `y`.
- **Real-Time Aggregation:** Calculate the average account balance and transaction duration aggregated by `job` category in real time using Spark Structured Streaming.
- **Display Output:** Continuously update and display the aggregated results via an in-memory streaming table sink.

In [6]:
# Task 1.1: Define Static Schema for Streaming Ingestion
transaction_schema = StructType([
    StructField("age", IntegerType(), True),
    StructField("job", StringType(), True),
    StructField("marital", StringType(), True),
    StructField("education", StringType(), True),
    StructField("default", StringType(), True),
    StructField("balance", DoubleType(), True),
    StructField("housing", StringType(), True),
    StructField("loan", StringType(), True),
    StructField("contact", StringType(), True),
    StructField("day", IntegerType(), True),
    StructField("month", StringType(), True),
    StructField("duration", DoubleType(), True),
    StructField("campaign", IntegerType(), True),
    StructField("pdays", IntegerType(), True),
    StructField("previous", IntegerType(), True),
    StructField("poutcome", StringType(), True),
    StructField("y", StringType(), True),
    StructField("timestamp", TimestampType(), True)
])

# Function to write timestamped micro-batch chunks into the stream directory
import pandas as pd
raw_bank_df = pd.read_csv("bank.csv")
chunk_size = 300

def emit_streaming_chunk(chunk_idx):
    start_idx = chunk_idx * chunk_size
    end_idx = min(start_idx + chunk_size, len(raw_bank_df))
    chunk_df = raw_bank_df.iloc[start_idx:end_idx].copy()
    chunk_df["timestamp"] = datetime.now()
    chunk_path = f"{STREAM_DATA_DIR}/chunk_{chunk_idx:03d}.csv"
    chunk_df.to_csv(chunk_path, index=False)
    return len(chunk_df)

# Seed initial batch
emit_streaming_chunk(0)
custom_display("Initial micro-batch chunk emitted into streaming pipeline!")

┌──────────────────────────────────────────────────────────────┐
│  Initial micro-batch chunk emitted into streaming pipeline!  │
└──────────────────────────────────────────────────────────────┘


In [9]:
# Task 1.2: Build Structured Streaming Reader & Real-Time Job Aggregations
streaming_raw = spark.readStream \
    .schema(transaction_schema) \
    .option("header", "true") \
    .option("maxFilesPerTrigger", 1) \
    .csv(STREAM_DATA_DIR)

# Real-time aggregation by job category
job_realtime_agg = streaming_raw.groupBy("job").agg(
    F.count("*").alias("total_transactions"),
    F.round(F.avg("balance"), 2).alias("avg_account_balance"),
    F.round(F.avg("duration"), 2).alias("avg_duration_sec")
)

# Start Streaming Query to In-Memory Sink
query_job_agg = job_realtime_agg.writeStream \
    .format("memory") \
    .queryName("realtime_job_aggregations") \
    .outputMode("complete") \
    .start()

print("Streaming Query started! Feeding chunks to monitor real-time updates...")

# Emit 2 more chunks and query results
for i in [1, 2]:
    time.sleep(2)
    count = emit_streaming_chunk(i)
    time.sleep(2)
    print(f"\n--- Real-Time Aggregated Snapshot after Chunk #{i} ({count} txns) ---")
    spark.sql("SELECT * FROM realtime_job_aggregations ORDER BY total_transactions DESC LIMIT 7").show()

query_job_agg.stop()
custom_display("Streaming query completed")

Streaming Query started! Feeding chunks to monitor real-time updates...

--- Real-Time Aggregated Snapshot after Chunk #1 (300 txns) ---
+-------------+------------------+-------------------+----------------+
|          job|total_transactions|avg_account_balance|avg_duration_sec|
+-------------+------------------+-------------------+----------------+
|   management|               196|            1892.12|          288.12|
|  blue-collar|               179|            1081.69|          272.25|
|   technician|               165|            1308.33|          247.94|
|       admin.|                86|            1434.72|          232.26|
|     services|                70|             889.91|          209.53|
|self-employed|                48|            1784.85|          294.08|
|      retired|                48|            1879.29|          241.79|
+-------------+------------------+-------------------+----------------+


--- Real-Time Aggregated Snapshot after Chunk #2 (300 txns) ---
+----

## Task 2: Real-Time Machine Learning Model Predictions
- **Model Development:** Train a supervised machine learning classification model on historical `bank.csv` data using Spark MLlib (`LogisticRegression` / `RandomForestClassifier`).
- **Pipeline Integration:** Embed the fitted ML pipeline into the live Spark Streaming data flow.
- **Real-Time Scoring:** Score incoming transaction batches on-the-fly and output individual predictions (`prediction` and probability) alongside the client profile.

In [10]:
# Task 2.1: Train Spark MLlib Classification Pipeline on Historical Data
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier

# Load historical static dataset for training
historical_df = spark.read.csv("bank.csv", header=True, inferSchema=True, sep=",") \
    .withColumn("balance", F.col("balance").cast(DoubleType())) \
    .withColumn("duration", F.col("duration").cast(DoubleType())) \
    .withColumn("age", F.col("age").cast(DoubleType())) \
    .withColumn("label", F.when(F.col("y") == "yes", 1.0).otherwise(0.0))

cat_cols = ['job', 'marital', 'education', 'housing', 'loan', 'contact']
num_cols = ['age', 'balance', 'duration']

# Pipeline Transformers
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_vec") for c in cat_cols]
assembler = VectorAssembler(inputCols=num_cols + [f"{c}_vec" for c in cat_cols], outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=False)

rf_classifier = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=6, seed=42)

# Build and fit complete end-to-end pipeline
ml_pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, rf_classifier])
trained_streaming_model = ml_pipeline.fit(historical_df)
custom_display("Machine Learning Pipeline successfully trained on historical banking data!")

┌────────────────────────────────────────────────────────────────────────┐
│  Machine Learning Pipeline successfully trained on historical banking  │
│  data!                                                                 │
└────────────────────────────────────────────────────────────────────────┘


In [11]:
# Task 2.2: Apply Trained Model to Live Structured Streaming Data
# Stream incoming records and transform them using the fitted pipeline model
streaming_stream_records = spark.readStream \
    .schema(transaction_schema) \
    .option("header", "true") \
    .option("maxFilesPerTrigger", 1) \
    .csv(STREAM_DATA_DIR)

# Real-time scoring through the trained Spark ML pipeline
scored_stream = trained_streaming_model.transform(streaming_stream_records) \
    .select(
        F.col("timestamp"),
        F.col("age"),
        F.col("job"),
        F.col("balance"),
        F.col("duration"),
        F.col("y").alias("actual_subscribed"),
        F.when(F.col("prediction") == 1.0, "Subscribed (1)").otherwise("No (0)").alias("predicted_subscription"),
        F.round(F.col("prediction"), 0).alias("prediction_code")
    )

# Write real-time inference stream to memory sink
query_predictions = scored_stream.writeStream \
    .format("memory") \
    .queryName("realtime_model_predictions") \
    .outputMode("append") \
    .start()

print("Real-Time Scoring Engine active! Ingesting and predicting incoming transactions...")

# Emit a new batch and inspect live predictions
time.sleep(2)
emit_streaming_chunk(3)
time.sleep(3)

print("\n--- LIVE REAL-TIME PREDICTIONS ON STREAMING TRANSACTIONS ---")
spark.sql("""
    SELECT timestamp, age, job, balance, duration, actual_subscribed, predicted_subscription
    FROM realtime_model_predictions
    ORDER BY timestamp DESC
    LIMIT 10
""").show(truncate=False)

query_predictions.stop()
custom_display("Real-time prediction pipeline completed")

Real-Time Scoring Engine active! Ingesting and predicting incoming transactions...

--- LIVE REAL-TIME PREDICTIONS ON STREAMING TRANSACTIONS ---
+--------------------------+---+------------+-------+--------+-----------------+----------------------+
|timestamp                 |age|job         |balance|duration|actual_subscribed|predicted_subscription|
+--------------------------+---+------------+-------+--------+-----------------+----------------------+
|2026-10-25 08:54:08.776133|33 |technician  |5235.0 |67.0    |no               |No (0)                |
|2026-10-25 08:54:08.776133|34 |entrepreneur|-139.0 |492.0   |no               |No (0)                |
|2026-10-25 08:54:08.776133|30 |technician  |3.0    |45.0    |no               |No (0)                |
|2026-10-25 08:54:08.776133|58 |technician  |409.0  |643.0   |no               |No (0)                |
|2026-10-25 08:54:08.776133|34 |admin.      |35.0   |76.0    |no               |No (0)                |
|2026-10-25 08:54:08.77

## Task 3: Window Operations and Trend Analysis
- **Window Operations:** Implement event-time windowing in Spark Structured Streaming using sliding/tumbling windows (e.g., 10-second and 30-second intervals).
- **Metric Tracking:** Compute and display transaction frequency count and average account balance dynamically within each time window.
- **Trend Discussion:** Analyze windowed velocity, volume spikes, and balance shifts observed across micro-batches.

In [13]:
# Task 3: Window Operations & Dynamic Temporal Trend Analysis
streaming_window_raw = spark.readStream \
    .schema(transaction_schema) \
    .option("header", "true") \
    .option("maxFilesPerTrigger", 1) \
    .csv(STREAM_DATA_DIR)

# 10-second tumbling window aggregation
windowed_trends = streaming_window_raw \
    .groupBy(
        F.window(F.col("timestamp"), "10 seconds")
    ).agg(
        F.count("*").alias("window_txn_count"),
        F.round(F.avg("balance"), 2).alias("window_avg_balance"),
        F.round(F.avg("duration"), 2).alias("window_avg_duration")
    )

query_windows = windowed_trends.writeStream \
    .format("memory") \
    .queryName("windowed_banking_trends") \
    .outputMode("complete") \
    .start()

# Stream micro-batches across distinct timestamps
for i in [4, 5]:
    time.sleep(1)
    emit_streaming_chunk(i)
    time.sleep(2)

print("\n--- REAL-TIME WINDOW OPERATIONS & VELOCITY TRENDS (10-SEC WINDOWS) ---")
spark.sql("""
    SELECT
        window.start AS window_start,
        window.end AS window_end,
        window_txn_count,
        window_avg_balance,
        window_avg_duration
    FROM windowed_banking_trends
    ORDER BY window_start DESC
""").show(truncate=False)

query_windows.stop()
custom_display("Window operations completed")


--- REAL-TIME WINDOW OPERATIONS & VELOCITY TRENDS (10-SEC WINDOWS) ---
+-------------------+-------------------+----------------+------------------+-------------------+
|window_start       |window_end         |window_txn_count|window_avg_balance|window_avg_duration|
+-------------------+-------------------+----------------+------------------+-------------------+
|2026-10-25 08:54:40|2026-10-25 08:54:50|600             |1258.5            |261.74             |
|2026-10-25 08:54:00|2026-10-25 08:54:10|300             |1456.96           |246.74             |
|2026-10-25 08:53:10|2026-10-25 08:53:20|600             |1508.22           |265.61             |
|2026-10-25 08:52:10|2026-10-25 08:52:20|300             |1449.9            |263.32             |
+-------------------+-------------------+----------------+------------------+-------------------+

┌───────────────────────────────┐
│  Window operations completed  │
└───────────────────────────────┘


### Analysis of Observed Streaming Trends
1. **Transaction Velocity & Stability:** The rolling window metrics demonstrate consistent micro-batch processing rates (~300 transactions per window interval).
2. **Liquidity Dynamics:** Average client balance remains steady around €1,300–€1,450 across temporal windows, indicating that high-volume campaign batches do not exhibit extreme liquidity skew.
3. **Operational Utility:** Windowed aggregations allow bank risk controllers to detect sudden volume surges or liquidity drops in near-real-time without querying massive historical databases.

## Task 4: Handling Late and Out-of-Order Data via Watermarking
- **Watermarking Strategy:** Implement `.withWatermark("timestamp", "15 seconds")` to manage out-of-order and delayed transaction events.
- **State Cleanup:** Ensure Spark automatically purges old aggregation state from memory once event-time falls behind the watermark boundary.
- **Technical Explanation:** Discuss how watermarking prevents out-of-memory (OOM) failures and ensures accuracy in production stream analytics.

In [14]:
# Task 4: Watermarking Implementation for Late & Out-of-Order Data
streaming_watermark_raw = spark.readStream \
    .schema(transaction_schema) \
    .option("header", "true") \
    .option("maxFilesPerTrigger", 1) \
    .csv(STREAM_DATA_DIR)

# Define 15-second watermark with 10-second tumbling windows
watermarked_stream = streaming_watermark_raw \
    .withWatermark("timestamp", "15 seconds") \
    .groupBy(
        F.window(F.col("timestamp"), "10 seconds"),
        F.col("y")
    ).agg(
        F.count("*").alias("txn_count"),
        F.round(F.avg("balance"), 2).alias("avg_balance")
    )

query_watermark = watermarked_stream.writeStream \
    .format("memory") \
    .queryName("watermarked_late_data_monitor") \
    .outputMode("complete") \
    .start()

# Stream another batch
time.sleep(1)
emit_streaming_chunk(6)
time.sleep(2)

print("\n--- WATERMARKED STREAMING ENGINE: AGGREGATED METRICS ---")
spark.sql("""
    SELECT
        window.start,
        window.end,
        y AS subscription_status,
        txn_count,
        avg_balance
    FROM watermarked_late_data_monitor
    ORDER BY window.start DESC
    LIMIT 6
""").show(truncate=False)

query_watermark.stop()
print("Watermarking run succesfully")


--- WATERMARKED STREAMING ENGINE: AGGREGATED METRICS ---
+-------------------+-------------------+-------------------+---------+-----------+
|start              |end                |subscription_status|txn_count|avg_balance|
+-------------------+-------------------+-------------------+---------+-----------+
|2026-10-25 08:54:40|2026-10-25 08:54:50|no                 |263      |1241.45    |
|2026-10-25 08:54:40|2026-10-25 08:54:50|yes                |37       |1677.59    |
|2026-10-25 08:54:00|2026-10-25 08:54:10|no                 |262      |1462.52    |
|2026-10-25 08:54:00|2026-10-25 08:54:10|yes                |38       |1418.61    |
|2026-10-25 08:53:10|2026-10-25 08:53:20|no                 |535      |1407.51    |
|2026-10-25 08:53:10|2026-10-25 08:53:20|yes                |65       |2337.18    |
+-------------------+-------------------+-------------------+---------+-----------+

Watermarking run succesfully


### Watermarking in Streaming ML
* **The Problem:** In production banking systems, mobile transactions or ATM events often arrive delayed or out-of-order due to network latency or intermittent device connectivity. Without constraints, the streaming engine must maintain state for all historical windows indefinitely, causing **executor out-of-memory (OOM) crashes**.
* **Watermark Approach (`withWatermark`):**
  $$\text{Watermark Threshold} = \max(\text{Event Time}) - \text{Allowed Latency Delay (15s)}$$
  Any incoming transaction whose timestamp is older than this moving boundary is safely dropped as too late to alter window states.
* **Accuracy & System Stability:** Watermarking bounds state memory while guaranteeing that legitimate in-flight delays are incorporated into rolling aggregations before windows finalize.

# Conclusion & Pipeline Summary

- **1. Structured Streaming:** Successfully constructed and evaluated real-time Spark micro-batch pipelines to monitor key job aggregates and client balance dynamics on-the-fly.
- **2. Real-Time ML serving:** Designed and trained an end-to-end classification pipeline featuring `StringIndexer`, `OneHotEncoder`, `VectorAssembler`, and `RandomForestClassifier` on historical bank data, applying it to score stream micro-batches in real time.
- **3. Temporal Windows & Trend Analysis:** Leveraged Spark's tumbling event-time windows (10-second intervals) to dynamically compute transaction velocity trends, volume spikes, and resource consumption patterns.
- **4. Stream Resilience via Watermarking:** Integrated a 15-second watermark boundary (`withWatermark`) on incoming event timestamps to manage late and out-of-order data, systematically purging stale window state to avoid memory overheads and OOM errors in production.

## ***End of Spark Streaming !***